# T2 — Jogo da Velha

Neste notebook vamos acompanhar a rede, o algoritmo genético e os resultados do treinamento. A MLP escolhe uma casa e o AG vai alterando os pesos conforme o desempenho das redes contra o Minimax. Não usamos backpropagation.

No VS Code, selecione o ambiente `.venv` em **Selecionar Kernel** e execute as células na ordem. As primeiras mostram o experimento que já está salvo. A parte de novo treinamento começa depois, com a configuração que podemos alterar.

Antes de executar, instale as dependências no terminal da raiz do projeto:

```bash
python3 -m pip install -r requirements-notebook-t2.txt
```

A interface continua disponível por `python3 app_t2.py`. O notebook serve para os experimentos e a explicação das etapas, como no T1.

In [ ]:
from pathlib import Path
from dataclasses import asdict
import sys
import json
import inspect
from html import escape
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Code, HTML, display

# O notebook pode ser aberto pela raiz do repositório ou pela pasta t2.
pasta_atual = Path.cwd().resolve()
raiz = next((p for p in [pasta_atual, *pasta_atual.parents] if (p / "app_t2.py").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Abra o notebook dentro da pasta do repositório.")
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from t2.algoritmos.mlp.mlp import Network
from t2.algoritmos.genetico.genetico import Config, Trainer, crossover
from t2.algoritmos.minimax.minimax import minimax, opponent_move
from t2.treinamento import REWARDS, evaluate, play

print("Projeto:", raiz.name)

In [ ]:
def mostrar_tabela(colunas, linhas):
    cabecalho = "".join(f"<th style='padding:8px;text-align:left'>{escape(str(c))}</th>" for c in colunas)
    corpo = "".join("<tr>" + "".join(f"<td style='padding:8px;border-top:1px solid #ddd'>{escape(str(v))}</td>" for v in linha) + "</tr>" for linha in linhas)
    display(HTML(f"<table><thead><tr>{cabecalho}</tr></thead><tbody>{corpo}</tbody></table>"))


def mostrar_evolucao(historico):
    geracoes = [h["generation"] for h in historico]
    fig, eixos = plt.subplots(1, 2, figsize=(11, 3.5))
    eixos[0].plot(geracoes, [h["best"] for h in historico], label="Melhor aptidão")
    eixos[0].plot(geracoes, [h["mean"] for h in historico], label="Média da população")
    eixos[0].set(xlabel="Geração", ylabel="Aptidão")
    eixos[0].legend()
    eixos[1].plot(geracoes, [100*h["valid_rate"] for h in historico])
    eixos[1].set(xlabel="Geração", ylabel="Escolhas válidas da melhor rede (%)", ylim=(0, 100))
    inicio_dificil = next((h["generation"] for h in historico if h["phase"] == "medio+dificil"), None)
    if inicio_dificil is not None:
        for eixo in eixos:
            eixo.axvline(inicio_dificil, color="gray", linestyle="--", linewidth=1)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

## Experimento que já está no projeto

Podemos consultar o treino salvo antes de executar outro. Os arquivos abaixo são os mesmos usados no relatório: população de 80, camada oculta com 9 neurônios, 250 gerações e semente 42. A linha pontilhada no gráfico marca a entrada das partidas no difícil.

A aptidão muda de conjunto de avaliação nessa passagem de fase. Por isso, uma diferença na curva nesse ponto não deve ser lida automaticamente como piora ou melhora da rede.

In [ ]:
arquivo_rede = raiz / "t2" / "resultados" / "rede-t2.json"
arquivo_avaliacao = raiz / "t2" / "resultados" / "avaliacao.json"
rede_salva = json.loads(arquivo_rede.read_text())
avaliacao_salva = json.loads(arquivo_avaliacao.read_text())
mostrar_tabela(["Parâmetro", "Valor"], list(rede_salva["config"].items()))
mostrar_evolucao(rede_salva["history"])

antes = avaliacao_salva["baseline"]["medium"]
depois = avaliacao_salva["final"]["medium"]
mostrar_tabela(["Teste médio: 100 partidas", "Primeira geração", "Última geração"], [
    ["Vitórias", antes["vitoria"], depois["vitoria"]],
    ["Empates", antes["empate"], depois["empate"]],
    ["Derrotas", antes["derrota"], depois["derrota"]],
    ["Acurácia de decisões", f'{100*antes["accuracy"]:.1f}%', f'{100*depois["accuracy"]:.1f}%'],
    ["Escolhas válidas", f'{100*antes["valid_rate"]:.1f}%', f'{100*depois["valid_rate"]:.1f}%'],
    ["Escolhas ocupadas", antes["invalid"], depois["invalid"]],
])

## Configuração de um novo treino

Daqui em diante vamos criar outra população. Os valores iniciais abaixo são os mesmos do experimento do relatório. Para testar outra configuração, altere esta célula e execute novamente a partir dela.

O treino usa várias partidas no médio. Depois dos primeiros 40% das gerações, acrescenta duas no difícil, alternando quem começa. A parada acontece pelo limite de gerações ou por falta de melhora na segunda fase.

In [ ]:
config = Config(
    population=80,
    hidden=9,
    generations=250,
    medium_games=24,
    elite=2,
    tournament=2,
    mutation=0.08,
    sigma=0.20,
    alpha=0.5,
    patience=60,
    seed=42,
)
config.validate()
print("Parâmetros por rede:", Network.size(config.hidden))

## Rede neural

A entrada representa as casas do tabuleiro. A própria peça vale 1, a adversária vale -1 e uma casa vazia vale 0. A rede aplica `h = tanh(x @ W1 + b1)` na camada oculta e `y = h @ W2 + b2` na saída.

O arquivo abaixo é a implementação usada tanto neste notebook quanto no jogo. Assim, a explicação do experimento e a interface usam a mesma rede. Ao executar a célula, podemos ler a classe e acompanhar a separação dos pesos e dos bias.

In [ ]:
display(Code(inspect.getsource(Network), language="python"))

## Cruzamento e avaliação

Cada indivíduo guarda os pesos de uma rede. O cruzamento é BLX-alpha, com `filho_i = pai_i + beta_i*(mae_i-pai_i)` e `beta_i ~ U(-alpha,1+alpha)`. Depois pode ocorrer a mutação.

A aptidão é `(3*vitórias + empates - 3*derrotas - 2*escolhas_inválidas) / partidas`. A escolha ocupada é registrada mesmo quando o programa usa a próxima saída válida para continuar a partida.

In [ ]:
display(Code(inspect.getsource(crossover), language="python"))
mostrar_tabela(["Resultado", "Recompensa"], list(REWARDS.items()))

## População inicial

Vamos criar a população com os pesos aleatórios entre -1 e 1. Cada linha é um cromossomo completo. Com a topologia inicial, são 180 parâmetros por indivíduo.

In [ ]:
treinador = Trainer(config)
print("Formato da população:", treinador.population.shape)
mostrar_tabela(["Gene", "Peso do primeiro indivíduo"], [
    [i, float(valor)] for i, valor in enumerate(treinador.population[0][:10])
])

## Primeira geração

Esta célula avalia a população inicial. Depois dela, já conseguimos olhar os resultados dos indivíduos e guardar a avaliação da melhor rede dessa geração para comparar no final.

In [ ]:
estado = treinador.step()
avaliacao_inicial = {
    "medium": evaluate(treinador.best, 100, False, 1000000, True),
    "hard": evaluate(treinador.best, 0, True, 2000000, True),
}
mostrar_tabela(["Indivíduo", "Aptidão", "Vitórias", "Empates", "Derrotas", "Inválidas"], [
    [i+1, r["fitness"], r["vitoria"], r["empate"], r["derrota"], r["invalid"]]
    for i, r in enumerate(treinador.ranked)
])

## Restante do treinamento

Agora repetimos as gerações. A classe `Trainer` preserva a elite, escolhe os pais por torneio e aplica cruzamento e mutação. A saída abaixo mostra o andamento a cada 25 gerações.

Para conferir como cada etapa está implementada, o código fica em `t2/algoritmos/genetico/genetico.py`. O notebook chama esse mesmo código, sem uma segunda implementação do AG.

In [ ]:
while not treinador.stopped:
    treinador.step()
    if treinador.generation % 25 == 0:
        ultima = treinador.history[-1]
        print(f'Geração {treinador.generation}: aptidão {ultima["best"]:.3f}, fase {ultima["phase"]}')
print("Treino encerrado:", treinador.stopped)
mostrar_evolucao(treinador.history)

## Teste da rede treinada

Usamos as sementes reservadas, fora da seleção do AG. A acurácia conta as escolhas originais que pertencem ao conjunto de ações ótimas do Minimax, divididas pelas decisões da rede. Uma escolha ocupada conta como erro.

Vitórias, empates e derrotas ficam separados. No difícil, são duas partidas determinísticas, uma para cada jogador inicial. Isso não cobre todas as sequências possíveis de jogadas ótimas.

In [ ]:
avaliacao_final = {
    "medium": evaluate(treinador.best, 100, False, 1000000, True),
    "hard": evaluate(treinador.best, 0, True, 2000000, True),
}
mostrar_tabela(["Modo", "Partidas", "Vitórias", "Empates", "Derrotas", "Inválidas", "Acurácia"], [
    [modo, r["games"], r["vitoria"], r["empate"], r["derrota"], r["invalid"], f'{100*r["accuracy"]:.1f}%']
    for modo, r in avaliacao_final.items()
])

## Uma partida, jogada por jogada

Aqui podemos mudar o modo do adversário e quem começa. A tabela permite conferir a escolha original da rede, a casa usada e a recompensa registrada. As casas são mostradas a partir de 1 para facilitar a leitura.

In [ ]:
modo = "dificil"
rede_comeca = True
partida = play(treinador.best, modo, rede_comeca, 1000000, trace=True, measure=True)
mostrar_tabela(["Jogada", "Jogador", "Escolha original", "Casa executada", "Recompensa"], [
    [i+1, e["actor"], e["raw"]+1 if e["raw"] is not None else "—", e["move"]+1, e["reward"]]
    for i, e in enumerate(partida["events"])
])
print("Resultado para a rede:", partida["outcome"])
print("Escolhas inválidas:", partida["invalid"])
print("Pontuação da partida:", partida["score"])

## Guardar o experimento do notebook

Os arquivos abaixo têm nomes próprios para não substituir a rede e a avaliação que já acompanham o relatório. Depois de executar esta célula, abra o jogo, clique em **Carregar rede** e escolha `t2/resultados/rede-notebook.json`.

In [ ]:
pasta_saida = raiz / "t2" / "resultados"
pasta_saida.mkdir(parents=True, exist_ok=True)
rede_notebook = {
    "version": 1,
    "hidden": treinador.best.hidden,
    "weights": treinador.best.weights.tolist(),
    "config": asdict(config),
    "history": treinador.history,
}
relatorio_notebook = {
    "config": asdict(config),
    "generations": treinador.generation,
    "stop": treinador.stopped,
    "baseline": avaliacao_inicial,
    "final": avaliacao_final,
}
(pasta_saida / "rede-notebook.json").write_text(json.dumps(rede_notebook, ensure_ascii=False, indent=2))
(pasta_saida / "avaliacao-notebook.json").write_text(json.dumps(relatorio_notebook, ensure_ascii=False, indent=2))
print("Rede salva:", pasta_saida / "rede-notebook.json")
print("Avaliação salva:", pasta_saida / "avaliacao-notebook.json")

## O que observar na análise

Compare o começo e o fim do treino, mas olhe também as escolhas inválidas. Se a rede ainda depende da recuperação para jogar em uma casa livre, isso precisa aparecer na explicação dos resultados.

Podemos variar a semente e a camada oculta para novos experimentos. Se os parâmetros forem escolhidos olhando repetidamente o mesmo teste, depois será preciso outra avaliação para conferir o resultado dessas escolhas.